<a href="https://colab.research.google.com/github/luandeferreira/Trabalho-Final-IA-Controle-HVAC-/blob/main/Trabalho_Final_IA_Controle_HVAC_Integrado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CÉLULA 1: Instalação e Importações
!pip install scikit-fuzzy

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
import skfuzzy as fuzz
from skfuzzy import control as ctrl

In [ ]:
# CÉLULA 2: Carregamento do Dataset
# Lendo o arquivo CSV que foi upado na sessão do Colab
df = pd.read_csv('HVAC_Dynamic_Fuzzy_PID_2017_with_Target.csv')

# Exibe as primeiras linhas para checar se as colunas foram carregadas corretamente
df.head()

In [ ]:
# CÉLULA 3: Treinamento do Machine Learning
# 1. Definindo Features (X) e Target (y)
X = df[['Temperature_C', 'Humidity_%', 'Occupancy_Count']]
y = df['User_Comfort_Index']

# 2. Dividindo os dados em Treino (80%) e Teste (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Inicializando e treinando o classificador por Árvore de Decisão
modelo_ml = DecisionTreeClassifier(max_depth=5, random_state=42)
modelo_ml.fit(X_train, y_train)

print("Modelo de Machine Learning treinado com sucesso!")

In [ ]:
# CÉLULA 4: Configuração das Variáveis Fuzzy
# Antecedente 1: Saída do ML (Índice de Conforto de 0 a 10)
conforto_previsto = ctrl.Antecedent(np.arange(0, 11, 1), 'conforto_previsto')

# Antecedente 2: Variável externa do Dataset (Temperatura Externa de 0°C a 40°C)
temp_externa = ctrl.Antecedent(np.arange(0, 41, 1), 'temp_externa')

# Consequente: Variável de controle (Potência do Ar-Condicionado de 0% a 100%)
potencia_hvac = ctrl.Consequent(np.arange(0, 101, 1), 'potencia_hvac')

# Mapeamento de Pertinência (Fuzzificação)
# Usando formas trapezoidais (trapmf) para as pontas e triangulares (trimf) para o meio
conforto_previsto['desconfortavel'] = fuzz.trapmf(conforto_previsto.universe, [0, 0, 3, 5])
conforto_previsto['aceitavel'] = fuzz.trimf(conforto_previsto.universe, [3, 5, 7])
conforto_previsto['confortavel'] = fuzz.trapmf(conforto_previsto.universe, [5, 7, 10, 10])

temp_externa['fria'] = fuzz.trapmf(temp_externa.universe, [0, 0, 15, 20])
temp_externa['amena'] = fuzz.trimf(temp_externa.universe, [15, 22, 28])
temp_externa['quente'] = fuzz.trapmf(temp_externa.universe, [25, 30, 40, 40])

potencia_hvac['baixa'] = fuzz.trimf(potencia_hvac.universe, [0, 20, 40])
potencia_hvac['media'] = fuzz.trimf(potencia_hvac.universe, [30, 50, 70])
potencia_hvac['alta'] = fuzz.trimf(potencia_hvac.universe, [60, 80, 100])

In [ ]:
# CÉLULA 5: Base de Regras Infericiais (COMPLETA)
regra1 = ctrl.Rule(conforto_previsto['desconfortavel'] & temp_externa['quente'], potencia_hvac['alta'])
regra2 = ctrl.Rule(conforto_previsto['desconfortavel'] & temp_externa['amena'], potencia_hvac['media'])
regra3 = ctrl.Rule(conforto_previsto['desconfortavel'] & temp_externa['fria'], potencia_hvac['alta'])

regra4 = ctrl.Rule(conforto_previsto['aceitavel'] & temp_externa['quente'], potencia_hvac['media'])
regra5 = ctrl.Rule(conforto_previsto['aceitavel'] & temp_externa['amena'], potencia_hvac['baixa'])
regra6 = ctrl.Rule(conforto_previsto['aceitavel'] & temp_externa['fria'], potencia_hvac['baixa'])

regra7 = ctrl.Rule(conforto_previsto['confortavel'] & temp_externa['quente'], potencia_hvac['baixa'])
regra8 = ctrl.Rule(conforto_previsto['confortavel'] & temp_externa['amena'], potencia_hvac['baixa'])
regra9 = ctrl.Rule(conforto_previsto['confortavel'] & temp_externa['fria'], potencia_hvac['baixa'])

# Compilando o sistema de controle com a matriz completa de regras
sistema_controle = ctrl.ControlSystem([regra1, regra2, regra3, regra4, regra5, regra6, regra7, regra8, regra9])
simulador = ctrl.ControlSystemSimulation(sistema_controle)

In [ ]:
# CÉLULA 6: Execução Integrada do Pipeline
# Cenário simulado: Uma sala interna a 29°C, com 70% de umidade e 18 pessoas dentro.
# Fora do prédio, os sensores apontam que a temperatura externa é de 34°C.
dados_sensores_internos = pd.DataFrame([[29.0, 70.0, 18]], columns=['Temperature_C', 'Humidity_%', 'Occupancy_Count'])
temperatura_externa_atual = 34.0

# Etapa 1: A Árvore de Decisão analisa os dados internos e faz a predição
conforto_predito_ml = modelo_ml.predict(dados_sensores_internos)[0]
print(f"[ML] Índice de Conforto Estimado pelo modelo: {conforto_predito_ml}")

# Etapa 2: Alimentamos o Controlador Fuzzy com a resposta do ML e o dado externo
simulador.input['conforto_previsto'] = conforto_predito_ml
simulador.input['temp_externa'] = temperatura_externa_atual

# Etapa 3: O motor Fuzzy processa as regras e realiza a defuzzificação (Centroide)
simulador.compute()
potencia_final_hvac = simulador.output['potencia_hvac']

print(f"[FUZZY] Potência de atuação calculada para o ar-condicionado: {potencia_final_hvac:.2f}%")

In [ ]:
# CÉLULA 7: Visualização das Funções de Pertinência
import matplotlib.pyplot as plt

# Mostra como o Conforto = 6 ativou os gráficos
conforto_previsto.view(sim=simulador)

# Mostra a Temperatura = 34
temp_externa.view(sim=simulador)

# Mostra o cálculo final da Potência = 35% (área preenchida)
potencia_hvac.view(sim=simulador)

plt.show()

In [ ]:
# CÉLULA 8: Superfície de Controle 3D (Corrigida)
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# 1. Criar uma malha (grid) para cobrir todas as combinações das duas entradas
x_conforto = np.arange(0, 11, 1)
y_temp = np.arange(0, 41, 1)

x_mesh, y_mesh = np.meshgrid(x_conforto, y_temp)
z_potencia = np.zeros_like(x_mesh)

# 2. Avaliar o sistema fuzzy para cada ponto da malha espacial
for i in range(x_mesh.shape[0]):
    for j in range(x_mesh.shape[1]):
        simulador.input['conforto_previsto'] = x_mesh[i, j]
        simulador.input['temp_externa'] = y_mesh[i, j]

        try:
            simulador.compute()
            z_potencia[i, j] = simulador.output['potencia_hvac']
        except (KeyError, ValueError):
            # Caso o sistema caia em algum ponto indefinido, atribui potência mínima
            z_potencia[i, j] = 0.0

# 3. Configurar e plotar o gráfico tridimensional
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Criando a superfície com o mapeamento de cores 'viridis' (azul para potências baixas, amarelo para altas)
surf = ax.plot_surface(x_mesh, y_mesh, z_potencia, cmap='viridis',
                       edgecolor='none', alpha=0.85)

# Títulos e rótulos dos eixos em português
ax.set_title('Superfície de Controle 3D - Tomada de Decisão Integrada', fontsize=14, pad=20)
ax.set_xlabel('Conforto Estimado (Saída do ML)', fontsize=11, labelpad=10)
ax.set_ylabel('Temperatura Externa (°C)', fontsize=11, labelpad=10)
ax.set_zlabel('Potência Atribuída ao HVAC (%)', fontsize=11, labelpad=10)

# Adiciona uma barra de cores lateral para servir de legenda técnica
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='Potência (%)')

# Ajusta o ângulo de visão inicial para facilitar a interpretação tridimensional
ax.view_init(elev=25, azim=135)

plt.tight_layout()
plt.show()